##### Load and inspect

In [7]:
import scipy.sparse as sp
import numpy as np
from collections import Counter

# Set the base directory
DATA_DIR = '/Users/vincent/masters-thesis/X-Transformer/datasets/Eurlex-4K'

print("="*80)
print("EURLEX-4K DATASET OVERVIEW")
print("="*80)

# Load the data
Y_train = sp.load_npz(f'{DATA_DIR}/Y.trn.npz')
Y_test = sp.load_npz(f'{DATA_DIR}/Y.tst.npz')
X_train = sp.load_npz(f'{DATA_DIR}/X.trn.npz')
X_test = sp.load_npz(f'{DATA_DIR}/X.tst.npz')

print(f"\nTraining set:")
print(f"  Documents: {Y_train.shape[0]:,}")
print(f"  Labels: {Y_train.shape[1]:,}")
print(f"  Features (TF-IDF): {X_train.shape[1]:,}")

print(f"\nTest set:")
print(f"  Documents: {Y_test.shape[0]:,}")
print(f"  Labels: {Y_test.shape[1]:,}")
print(f"  Features (TF-IDF): {X_test.shape[1]:,}")

# Data type and sparsity
print(f"\nData characteristics:")
print(f"  Y_train sparsity: {100 * (1 - Y_train.nnz / (Y_train.shape[0] * Y_train.shape[1])):.4f}%")
print(f"  X_train sparsity: {100 * (1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])):.4f}%")

EURLEX-4K DATASET OVERVIEW

Training set:
  Documents: 15,449
  Labels: 3,956
  Features (TF-IDF): 186,104

Test set:
  Documents: 3,865
  Labels: 3,956
  Features (TF-IDF): 186,104

Data characteristics:
  Y_train sparsity: 99.8654%
  X_train sparsity: 99.8541%


##### Explore labels

In [8]:
print("\n" + "="*80)
print("LABEL INFORMATION")
print("="*80)

# Load label descriptions
with open(f'{DATA_DIR}/label_map.txt', 'r', encoding='utf-8') as f:
    label_map = [line.strip() for line in f.readlines()]

print(f"\nTotal labels in label_map.txt: {len(label_map)}")

# Separate numeric and text labels
import re
numeric_labels = [(i, l) for i, l in enumerate(label_map) if re.match(r'^\d+\.?\d*$', l)]
text_labels = [(i, l) for i, l in enumerate(label_map) if not re.match(r'^\d+\.?\d*$', l)]

print(f"  Numeric labels (likely IDs): {len(numeric_labels)}")
print(f"  Text labels (descriptions): {len(text_labels)}")

print(f"\nFirst 20 labels:")
for i in range(20):
    print(f"  {i:4d}: {label_map[i]}")

print(f"\nLast 20 labels:")
for i in range(len(label_map)-20, len(label_map)):
    print(f"  {i:4d}: {label_map[i]}")

print(f"\nRandom sample of text labels:")
import random
sample_indices = random.sample([i for i, _ in text_labels], 20)
for idx in sorted(sample_indices):
    print(f"  {idx:4d}: {label_map[idx]}")


LABEL INFORMATION

Total labels in label_map.txt: 3956
  Numeric labels (likely IDs): 20
  Text labels (descriptions): 3936

First 20 labels:
     0: 2164
     1: 2164.0
     2: 3485.0
     3: 4067
     4: 4067.0
     5: 5425.0
     6: 5641
     7: 5641.0
     8: 5759.0
     9: 7410.0
    10: 7813.0
    11: 7932.0
    12: 7936.0
    13: 7937.0
    14: 7950.0
    15: 7951.0
    16: 7953.0
    17: 7954.0
    18: 7985.0
    19: 7986.0

Last 20 labels:
  3936: xenophobia
  3937: yemen
  3938: yoghourt
  3939: yorkshire and humberside
  3940: young farmer
  3941: young person
  3942: young worker
  3943: youth employment
  3944: youth exchange scheme
  3945: youth movement
  3946: youth policy
  3947: youth unemployment
  3948: yugoslavia
  3949: zambia
  3950: zimbabwe
  3951: zinc
  3952: zoo
  3953: zoology
  3954: zoonosis
  3955: zootechnics

Random sample of text labels:
   296: auxiliary worker
   854: cultural event
   950: development aid
  1107: ec ombudsman
  1118: ec technical 

##### Analyze label distribution

In [9]:
print("\n" + "="*80)
print("LABEL DISTRIBUTION ANALYSIS")
print("="*80)

# Count how many times each label appears
label_counts_train = np.array(Y_train.sum(axis=0)).flatten()
label_counts_test = np.array(Y_test.sum(axis=0)).flatten()

print("\nLabel frequency statistics (Training set):")
print(f"  Min occurrences: {label_counts_train.min()}")
print(f"  Max occurrences: {label_counts_train.max()}")
print(f"  Mean occurrences: {label_counts_train.mean():.2f}")
print(f"  Median occurrences: {np.median(label_counts_train):.0f}")

# Head vs tail labels
print(f"\nLabel frequency distribution:")
tail_labels = np.sum(label_counts_train < 5)
print(f"  Tail labels (<5 occurrences): {tail_labels} ({100*tail_labels/len(label_counts_train):.1f}%)")
print(f"  Labels with 5-50 occurrences: {np.sum((label_counts_train >= 5) & (label_counts_train < 50))}")
print(f"  Labels with 50-500 occurrences: {np.sum((label_counts_train >= 50) & (label_counts_train < 500))}")
print(f"  Head labels (≥500 occurrences): {np.sum(label_counts_train >= 500)}")

# Top 20 most frequent labels
top_20_indices = np.argsort(label_counts_train)[-20:][::-1]
print(f"\nTop 20 most frequent labels (Training set):")
for rank, idx in enumerate(top_20_indices, 1):
    print(f"  {rank:2d}. Label {idx:4d} ({label_map[idx][:50]:50s}): {label_counts_train[idx]:5.0f} docs")

# Bottom 20 least frequent labels
bottom_20_indices = np.argsort(label_counts_train)[:20]
print(f"\nBottom 20 least frequent labels (Training set):")
for rank, idx in enumerate(bottom_20_indices, 1):
    if label_counts_train[idx] > 0:
        print(f"  {rank:2d}. Label {idx:4d} ({label_map[idx][:50]:50s}): {label_counts_train[idx]:5.0f} docs")


LABEL DISTRIBUTION ANALYSIS

Label frequency statistics (Training set):
  Min occurrences: 0
  Max occurrences: 1024
  Mean occurrences: 20.79
  Median occurrences: 5

Label frequency distribution:
  Tail labels (<5 occurrences): 1906 (48.2%)
  Labels with 5-50 occurrences: 1656
  Labels with 50-500 occurrences: 386
  Head labels (≥500 occurrences): 8

Top 20 most frequent labels (Training set):
   1. Label 1072 (ec agreement                                      ):  1024 docs
   2. Label 1857 (import                                            ):   934 docs
   3. Label 1769 (health control                                    ):   765 docs
   4. Label 3426 (state aid                                         ):   744 docs
   5. Label  233 (approximation of laws                             ):   712 docs
   6. Label 2641 (originating product                               ):   680 docs
   7. Label 1348 (european economic area                            ):   587 docs
   8. Label 1932 (informat

##### Analyze Multi-Label Characteristics

In [10]:
print("\n" + "="*80)
print("MULTI-LABEL CHARACTERISTICS")
print("="*80)

# Labels per document
labels_per_doc_train = np.array(Y_train.sum(axis=1)).flatten()
labels_per_doc_test = np.array(Y_test.sum(axis=1)).flatten()

print("\nLabels per document (Training set):")
print(f"  Min: {labels_per_doc_train.min():.0f}")
print(f"  Max: {labels_per_doc_train.max():.0f}")
print(f"  Mean: {labels_per_doc_train.mean():.2f}")
print(f"  Median: {np.median(labels_per_doc_train):.0f}")

# Distribution
print(f"\nDistribution of labels per document (Training):")
label_dist = Counter(labels_per_doc_train.astype(int))
for num_labels in sorted(label_dist.keys())[:20]:  # First 20
    count = label_dist[num_labels]
    print(f"  {num_labels:2d} labels: {count:5d} documents ({100*count/Y_train.shape[0]:.1f}%)")

if len(label_dist) > 20:
    print(f"  ... ({len(label_dist) - 20} more categories)")

print(f"\nTest set:")
print(f"  Mean labels per doc: {labels_per_doc_test.mean():.2f}")
print(f"  Median labels per doc: {np.median(labels_per_doc_test):.0f}")


MULTI-LABEL CHARACTERISTICS

Labels per document (Training set):
  Min: 1
  Max: 24
  Mean: 5.32
  Median: 6

Distribution of labels per document (Training):
   1 labels:     7 documents (0.0%)
   2 labels:   303 documents (2.0%)
   3 labels:  1112 documents (7.2%)
   4 labels:  2331 documents (15.1%)
   5 labels:  3867 documents (25.0%)
   6 labels:  6327 documents (41.0%)
   7 labels:   844 documents (5.5%)
   8 labels:   358 documents (2.3%)
   9 labels:   187 documents (1.2%)
  10 labels:    75 documents (0.5%)
  11 labels:    16 documents (0.1%)
  12 labels:     7 documents (0.0%)
  13 labels:     4 documents (0.0%)
  14 labels:     3 documents (0.0%)
  15 labels:     4 documents (0.0%)
  17 labels:     2 documents (0.0%)
  18 labels:     1 documents (0.0%)
  24 labels:     1 documents (0.0%)

Test set:
  Mean labels per doc: 5.30
  Median labels per doc: 5


##### Explore Raw Text

In [12]:
print("\n" + "="*80)
print("RAW TEXT EXPLORATION")
print("="*80)

# Load raw texts
with open(f'{DATA_DIR}/train_raw_texts.txt', 'r', encoding='utf-8', errors='ignore') as f:
    train_texts = f.readlines()

with open(f'{DATA_DIR}/test_raw_texts.txt', 'r', encoding='utf-8', errors='ignore') as f:
    test_texts = f.readlines()

print(f"\nNumber of training documents: {len(train_texts)}")
print(f"Number of test documents: {len(test_texts)}")

# Text length analysis
train_lengths = [len(text.split()) for text in train_texts]

print(f"\nText length statistics (in words):")
print(f"  Min: {min(train_lengths)}")
print(f"  Max: {max(train_lengths)}")
print(f"  Mean: {np.mean(train_lengths):.1f}")
print(f"  Median: {np.median(train_lengths):.0f}")

# Length distribution
print(f"\nText length distribution:")
bins = [0, 50, 100, 200, 500, 1000, 2000, 5000]
for i in range(len(bins)-1):
    count = sum(1 for l in train_lengths if bins[i] <= l < bins[i+1])
    print(f"  {bins[i]:5d}-{bins[i+1]:5d} words: {count:5d} docs ({100*count/len(train_lengths):.1f}%)")
count = sum(1 for l in train_lengths if l >= bins[-1])
print(f"  {bins[-1]:5d}+ words: {count:5d} docs ({100*count/len(train_lengths):.1f}%)")

print(f"\n" + "-"*80)
print("SAMPLE TRAINING DOCUMENTS:")
print("-"*80)

# Show 3 random documents
for i in random.sample(range(len(train_texts)), 3):
    print(f"\nDocument {i}:")
    print(f"  Labels: {np.where(Y_train[i].toarray()[0] == 1)[0]}")
    label_names = [label_map[idx] for idx in np.where(Y_train[i].toarray()[0] == 1)[0]]
    print(f"  Label names: {label_names}")
    print(f"  Text (first 300 chars):")
    print(f"    {train_texts[i][:300]}...")


RAW TEXT EXPLORATION

Number of training documents: 15449
Number of test documents: 3865

Text length statistics (in words):
  Min: 3
  Max: 189265
  Mean: 1240.6
  Median: 393

Text length distribution:
      0-   50 words:    30 docs (0.2%)
     50-  100 words:   253 docs (1.6%)
    100-  200 words:  3378 docs (21.9%)
    200-  500 words:  5177 docs (33.5%)
    500- 1000 words:  2735 docs (17.7%)
   1000- 2000 words:  1803 docs (11.7%)
   2000- 5000 words:  1379 docs (8.9%)
   5000+ words:   694 docs (4.5%)

--------------------------------------------------------------------------------
SAMPLE TRAINING DOCUMENTS:
--------------------------------------------------------------------------------

Document 5990:
  Labels: [ 444  482 1083 1757 3710]
  Label names: ['cable transport', 'carriage of passengers', 'ec conformity marking', 'harmonisation of standards', 'transport safety']
  Text (first 300 chars):
    direct ec european parliament council march relat cablewai instal design ca

##### Label Co-occurrence Analysis


In [14]:
print("\n" + "="*80)
print("LABEL CO-OCCURRENCE ANALYSIS")
print("="*80)

# Find most common label pairs
from itertools import combinations

print("\nAnalyzing label co-occurrences (this may take a moment)...")

label_pairs = Counter()
for i in range(min(1000, Y_train.shape[0])):  # Sample first 1000 docs
    doc_labels = np.where(Y_train[i].toarray()[0] == 1)[0]
    if len(doc_labels) >= 2:
        for pair in combinations(doc_labels, 2):
            label_pairs[tuple(sorted(pair))] += 1

print(f"\nTop 20 most common label pairs (in first 1000 documents):")
for rank, (pair, count) in enumerate(label_pairs.most_common(20), 1):
    l1, l2 = pair
    name1 = label_map[l1][:30]
    name2 = label_map[l2][:30]
    print(f"  {rank:2d}. ({l1:4d}, {l2:4d}) = ({name1:30s}, {name2:30s}): {count:3d} times")


LABEL CO-OCCURRENCE ANALYSIS

Analyzing label co-occurrences (this may take a moment)...

Top 20 most common label pairs (in first 1000 documents):
   1. (1072, 1348) = (ec agreement                  , european economic area        ):  25 times
   2. (1857, 2641) = (import                        , originating product           ):  23 times
   3. ( 782, 3426) = (control of state aid          , state aid                     ):  21 times
   4. (1769, 1857) = (health control                , import                        ):  16 times
   5. (1841, 2543) = (iceland                       , norway                        ):  16 times
   6. ( 192, 1769) = (animal disease                , health control                ):  16 times
   7. (1860, 3533) = (import licence                , tariff quota                  ):  15 times
   8. (1768, 1857) = (health certificate            , import                        ):  15 times
   9. (1769, 3819) = (health control                , veterinary inspection

##### Dataset Summary

In [15]:
print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)

print(f"""
EURLEX-4K Dataset Characteristics:

Documents:
  - Training: {Y_train.shape[0]:,}
  - Test: {Y_test.shape[0]:,}
  - Avg text length: {np.mean(train_lengths):.0f} words

Labels:
  - Total labels: {Y_train.shape[1]:,}
  - Text labels: {len(text_labels):,}
  - Numeric labels: {len(numeric_labels)}
  - Avg labels per doc: {labels_per_doc_train.mean():.2f}
  - Tail labels (<5 docs): {tail_labels} ({100*tail_labels/len(label_counts_train):.1f}%)

Features:
  - TF-IDF vocabulary size: {X_train.shape[1]:,}
  - Feature matrix sparsity: {100 * (1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])):.2f}%

Label Language:
  - English text descriptions: {len(text_labels):,}
  - Need translation for multilingual experiments
""")


DATASET SUMMARY

EURLEX-4K Dataset Characteristics:

Documents:
  - Training: 15,449
  - Test: 3,865
  - Avg text length: 1241 words

Labels:
  - Total labels: 3,956
  - Text labels: 3,936
  - Numeric labels: 20
  - Avg labels per doc: 5.32
  - Tail labels (<5 docs): 1906 (48.2%)

Features:
  - TF-IDF vocabulary size: 186,104
  - Feature matrix sparsity: 99.85%

Label Language:
  - English text descriptions: 3,936
  - Need translation for multilingual experiments

